# Import modules

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Set results path

In [ ]:
results_path = '/CCLE_res'

# Data

In [ ]:
rna = pd.read_csv('CCLE_transcriptomics_tpm_metabolism_related_genes.csv', index_col = 0 )
rna

In [ ]:
metabolites = pd.read_csv('CCLE_metabolites.csv', index_col = 0)
metabolites

In [ ]:
threshold = 0.2
nan_percentage = metabolites.isnull().mean()
filtered_metabolites = metabolites.loc[:, nan_percentage <= threshold]
metabolites = filtered_metabolites
metabolites

# Remove outliers

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer

def remove_outliers_isolationforest(data, contamination=0.05):
    """Removes outliers using Isolation Forest."""

    imputer = SimpleImputer(strategy='mean') 
    data_clean = imputer.fit_transform(data)

    iso = IsolationForest(contamination=contamination)
    yhat = iso.fit_predict(data_clean)
    filtered_data = data[yhat != -1]  
    return filtered_data


filtered_rna = remove_outliers_isolationforest(rna)
filtered_metabolites = remove_outliers_isolationforest(metabolites)

In [ ]:
rna = filtered_rna
metabolites = filtered_metabolites

In [ ]:
rna

# Data shuffling

In [ ]:
combined_data = pd.concat([rna, metabolites], axis=1)
shuffled_data = combined_data.sample(frac=1, random_state=0)
shuffled_rna = shuffled_data[rna.columns]
shuffled_metabolites = shuffled_data[metabolites.columns]

rna = shuffled_rna
metabolites = shuffled_metabolites

# Log2 transformation

In [ ]:
#CCLE data are already log2 transformed
# rna = rna.apply(lambda x: np.log2(x + 1))
# rna.head(3)

# Check data normality

In [ ]:
from scipy.stats import shapiro, kstest, anderson
import numpy as np

rna_data = rna.values.ravel()

# Shapiro-Wilk Test
statistic, p_value = shapiro(rna_data)
print(f'Shapiro-Wilk Test: Statistic={statistic}, p-value={p_value}')

# Kolmogorov-Smirnov Test
statistic, p_value = kstest(rna_data, 'norm')
print(f'Kolmogorov-Smirnov Test: Statistic={statistic}, p-value={p_value}')

# Anderson-Darling Test
result = anderson(rna_data, dist='norm')
print(f'Anderson-Darling Test: Statistic={result.statistic}, Critical Values={result.critical_values}')

In [ ]:
met_data = metabolites.values.ravel()

# Shapiro-Wilk Test
statistic, p_value = shapiro(met_data)
print(f'Shapiro-Wilk Test: Statistic={statistic}, p-value={p_value}')

# Kolmogorov-Smirnov Test
statistic, p_value = kstest(met_data, 'norm')
print(f'Kolmogorov-Smirnov Test: Statistic={statistic}, p-value={p_value}')

# Anderson-Darling Test
result = anderson(met_data, dist='norm')
print(f'Anderson-Darling Test: Statistic={result.statistic}, Critical Values={result.critical_values}')

# Data distribution

In [ ]:
# For RNA data
plt.figure(figsize=(10, 6))
sns.histplot(rna.values.ravel(), bins=500, kde=True)
plt.title("Distribution of RNA expression values")
plt.xlabel("Expression values")
plt.ylabel("Frequency")
plt.savefig(f'{results_path}/rna_sample_distribution.svg', bbox_inches = 'tight')
plt.show()

# For metabolite data
plt.figure(figsize=(10, 6))
sns.histplot(metabolites.values.ravel(), bins=50, kde=True)
plt.title("Distribution of Metabolite levels")
plt.xlabel("Metabolite levels")
plt.ylabel("Frequency")
plt.savefig(f'{results_path}/metabolite_sample_distribution.svg', bbox_inches = 'tight')
plt.show()

# Splitting data

In [ ]:
metabolites = metabolites.dropna(axis=1, how='all')
metabolites

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(rna, metabolites, test_size=0.2, random_state=0)

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer_train = SimpleImputer(strategy='mean')
y_train_imputed = imputer_train.fit_transform(y_train)

In [ ]:
imputer_test = SimpleImputer(strategy='mean')
y_test_imputed = imputer_test.fit_transform(y_test)

In [ ]:
y_train = pd.DataFrame(y_train_imputed, columns=y_train.columns, index=y_train.index)
y_test = pd.DataFrame(y_test_imputed, columns=y_test.columns, index=y_test.index)

In [ ]:
if y_train.isnull().any().any():
    print("DataFrame contains NaN values")
else:
    print("DataFrame does not contain NaN values")

In [ ]:
if y_test.isnull().any().any():
    print("DataFrame contains NaN values")
else:
    print("DataFrame does not contain NaN values")

In [ ]:
y_train_features = y_train.columns
y_train_features

In [ ]:
import pickle

# Save the list to a file
with open(f'{results_path}/y_train_features.pkl', 'wb') as f:
    pickle.dump(y_train_features, f)

# Import sklearn modules

In [ ]:
!pip install xgboost

In [ ]:
!pip install lightgbm

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import MultiTaskLasso
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge

# Scoring and plotting functions

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

def get_r2_mse_rmse_plot_pred(data_y_test, model_predictions, model_type):

  r2 = r2_score(data_y_test, model_predictions)
  mse = mean_squared_error(data_y_test, model_predictions)
  rmse = mse**0.5

  print(f"R-squared: {r2}")
  print(f"Mean Squared Error (MSE): {mse}")
  print(f"Root Mean Squared Error (RMSE): {rmse}")


  plt.figure(figsize=(8, 6))
  for i in range(data_y_test.shape[1]):
      plt.scatter(data_y_test.iloc[:, i], model_predictions[:, i], alpha=0.5)

  plt.plot([data_y_test.min().min(), data_y_test.max().max()], [data_y_test.min().min(), data_y_test.max().max()], 'k--', lw=2, label='Ideal')

  plt.xlabel("Actual Values")
  plt.ylabel("Predicted Values")
  plt.title(f"Predicted vs. Actual Values for Metabolites ({model_type})")
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper right')
  plt.grid(True)
  plt.savefig(f'{results_path}/{model_type}.svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

def plot_cv_scores(model_pipeline, model_type,cv_fold):

  imputer_metabolites = SimpleImputer(strategy='mean')
  metabolites_imputed = imputer_metabolites.fit_transform(metabolites)
  metabolites_imputed = pd.DataFrame(metabolites_imputed, columns=metabolites.columns, index=metabolites.index)

  scores = cross_val_score(model_pipeline, rna, metabolites_imputed, cv=cv_fold, scoring='neg_mean_squared_error', n_jobs = -1)

  print("Cross-validation scores:", scores)
  print("Average score:", scores.mean())

  scores = pd.DataFrame(scores)
  scores = scores.rename(columns = {0: 'Cross-validation scores'})
  scores['Cross-validation fold'] = [i + 1 for i in range(len(scores))]

  sns.lineplot(data = scores, x = 'Cross-validation fold', y = 'Cross-validation scores')
  plt.ylabel('Negative MSE')
  plt.title('Negative MSE fluctuation with CV-fold')
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
  plt.savefig(f'{results_path}/MSE vs CV ({model_type}).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
from sklearn.model_selection import KFold

def get_r2_cross_val(model, model_name):

  cv = KFold(n_splits=10, shuffle=True, random_state=0)

  scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='r2', n_jobs = -1)

  print("Cross-validation R-squared scores:", scores)
  print("Average R-squared score:", scores.mean())

  scores_df = pd.DataFrame({'Fold': range(1, len(scores) + 1), 'R-squared': scores})

  # Plot the scores
  plt.figure(figsize=(8, 6))
  sns.lineplot(x='Fold', y='R-squared', data=scores_df, marker='o')
  plt.title(f'Cross-validation R-squared scores for {model_name}')
  plt.xlabel('Fold')
  plt.ylabel('R-squared')
  plt.savefig(f'{results_path}/R2 vs CV ({model_name}).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error

def get_metrics(y_predicted):

  mse = mean_squared_error(y_test, y_predicted)
  print("MSE:", mse)

  from sklearn.metrics import mean_absolute_error
  mae = mean_absolute_error(y_test, y_predicted)
  print("MAE:", mae)

  from sklearn.metrics import median_absolute_error
  medae = median_absolute_error(y_test, y_predicted,)
  print("MedAE:", medae)

  def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

  mape = mean_absolute_percentage_error(y_test, y_predicted)
  print('Mean absolute percentage error:', mape)

In [ ]:
from scipy.stats import spearmanr

def get_spearman_correlation(model_predictions, model_name):

  correlation, p_value = spearmanr(X_test, model_predictions)

  correlation_df = pd.DataFrame({'Actual': y_test.values.flatten(), 'Predicted': model_predictions.flatten()})

  correlation, p_value = spearmanr(correlation_df['Actual'], correlation_df['Predicted'])

  correlation_df['Correlation'] = correlation
  correlation_df['P-value'] = p_value

  sns.regplot(x='Actual', y='Predicted', data=correlation_df, line_kws={'color': 'red'})
  plt.title(f'Spearman\'s Correlation ({model_name})')
  plt.xlabel('Actual Values')
  plt.ylabel('Predicted Values')

  plt.text(0.1, 0.9, f'Correlation: {correlation:.2f}\nP-value: {p_value:.2f}', transform=plt.gca().transAxes)
  plt.savefig(f'{results_path}/Spearmans Corr ({model_name}).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
#Checking for heteroscedasticity
!pip install statsmodels

import statsmodels.api as sm

def get_residuals_vs_pred_plot(model_predictions, model_name):

  residuals = (y_test - model_predictions).values.ravel()

  plt.figure(figsize=(8, 6))
  plt.scatter(model_predictions.ravel(), residuals, alpha=0.6)
  plt.axhline(y=0, color='red', linestyle='--', linewidth=1)
  plt.title(f"Residuals vs. Predicted Values ({model_name})")
  plt.xlabel("Predicted Values")
  plt.ylabel("Residuals")
  plt.savefig(f'{results_path}/Residuals vs Pred Values ({model_name}).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
def plot_r2_per_metabolite(model, model_name, X_test, y_test):
    """Plots the R2 score per metabolite for a given model in descending order."""

    y_pred = model.predict(X_test) 

    r2_scores = []  
    for i in range(y_test.shape[1]):  
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])  
        r2_scores.append(r2)  

    r2_df = pd.DataFrame({'Metabolite': y_test.columns, 'R2 Score': r2_scores})

    r2_df = r2_df.sort_values(by=['R2 Score'], ascending=False)

    plt.figure(figsize=(10, 6))  
    plt.bar(range(len(r2_df)), r2_df['R2 Score'])  
    plt.xticks()  
    plt.xlabel("Metabolite")
    plt.ylabel("R2 Score")
    plt.title(f"R2 Score per Metabolite ({model_name}) - Descending Order")
    plt.tight_layout()  
    plt.savefig(f'{results_path}/R2 per metabolite ({model_name}).svg', bbox_inches = 'tight')
    plt.show()

In [ ]:
def get_r2_per_metabolite(model, model_name, X_test, y_test):
    """Get the R2 score per metabolite for a given model in descending order."""

    y_pred = model.predict(X_test)  

    r2_scores = []  
    for i in range(y_test.shape[1]): 
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])  
        r2_scores.append(r2) 

    r2_df = pd.DataFrame({'Metabolite': y_test.columns, 'R2 Score': r2_scores})

    r2_df = r2_df.sort_values(by=['R2 Score'], ascending=False)

    return r2_df

In [ ]:
from scipy.stats import spearmanr

def plot_spearman_per_metabolite(model, model_name, X_test, y_test):
    """Plots the Spearman's correlation coefficient per metabolite for a given model."""

    y_pred = model.predict(X_test)  

    spearman_coeffs = []  
    for i in range(y_test.shape[1]):  
        coeff, _ = spearmanr(y_test.iloc[:, i], y_pred[:, i])  
        spearman_coeffs.append(coeff)  

    spearman_df = pd.DataFrame({'Metabolite': y_test.columns, 'Spearman Coefficient': spearman_coeffs})

    spearman_df = spearman_df.sort_values(by=['Spearman Coefficient'], ascending=False)

    plt.figure(figsize=(10, 6))  
    plt.bar(range(len(spearman_df)), spearman_df['Spearman# Set x-axis labels to metabolite names
    plt.xlabel("Metabolite")
    plt.ylabel("Spearman's Coefficient")
    plt.title(f"Spearman's Correlation Coefficient per Metabolite ({model_name}) - Descending Order")
    plt.tight_layout() 
    plt.savefig(f'{results_path}/Spearman per metabolite ({model_name}).svg', bbox_inches = 'tight')
    plt.show()

In [ ]:
def get_spearman_per_metabolite(model, model_name, X_test, y_test):
    """Get the Spearman's correlation coefficient per metabolite for a given model."""

    y_pred = model.predict(X_test)  

    spearman_coeffs = [] 
    for i in range(y_test.shape[1]):  
        coeff, _ = spearmanr(y_test.iloc[:, i], y_pred[:, i]) 
        spearman_coeffs.append(coeff) 

    spearman_df = pd.DataFrame({'Metabolite': y_test.columns, 'Spearman Coefficient': spearman_coeffs})

    spearman_df = spearman_df.sort_values(by=['Spearman Coefficient'], ascending=False)

    return spearman_df

In [ ]:
import joblib

def save_model(model, model_name):
    joblib.dump(model, f'{results_path}/{model_name}.pkl')

# Ridge

In [ ]:
pipeline_ridge = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('pca', PCA(0.95,)),
    ('model', Ridge(random_state = 0))
                          ])

In [ ]:
param_grid_ridge = {
    'model__alpha' : [4500, 5000, 6000, 6050, 6075, 6100, 6125, 6150, 6175, 6250, 6500, 7000]
}

In [ ]:
grid_search_ridge = GridSearchCV(
    pipeline_ridge,
    param_grid_ridge,
    cv = 5,
    scoring = 'neg_mean_squared_error',
    n_jobs = -1
)

grid_search_ridge.fit(X_train, y_train)

In [ ]:
print("Best hyperparameters:", grid_search_ridge.best_params_)
print("Best score:", grid_search_ridge.best_score_)

In [ ]:
best_ridge_model = grid_search_ridge.best_estimator_

In [ ]:
save_model(model = best_ridge_model, model_name = 'Ridge')

In [ ]:
ridge_predictions = best_ridge_model.predict(X_test)

In [ ]:
plot_cv_scores(model_pipeline = best_ridge_model, model_type = 'Ridge', cv_fold = 10)

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = ridge_predictions, model_type = 'Ridge')

In [ ]:
get_r2_cross_val(model = best_ridge_model, model_name = 'Ridge')

In [ ]:
get_metrics(y_predicted = ridge_predictions)

In [ ]:
get_spearman_correlation(model_predictions = ridge_predictions, model_name = 'Ridge')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = ridge_predictions, model_name = 'Ridge')

In [ ]:
plot_r2_per_metabolite(best_ridge_model, "Ridge", X_test, y_test)

In [ ]:
r2_ridge = get_r2_per_metabolite(best_ridge_model, 'Ridge', X_test, y_test)
r2_ridge

In [ ]:
plot_spearman_per_metabolite(best_ridge_model, 'Ridge', X_test, y_test)

In [ ]:
spearman_ridge = get_spearman_per_metabolite(best_ridge_model, 'Ridge', X_test, y_test)
spearman_ridge

In [ ]:
r2_ridge.to_csv(f'{results_path}/r2_ridge.csv', index = False)

In [ ]:
spearman_ridge.to_csv(f'{results_path}/spearman_ridge.csv', index = False)

# Multi-task Lasso

In [ ]:
pipeline_multi_lasso = Pipeline([
                                ('imputer', SimpleImputer(strategy='mean')),
                                ('scaler', StandardScaler()),
                                ('pca', PCA(0.95,)),
                                ('model', MultiTaskLasso(random_state = 0))
                                ])

In [ ]:
param_grid_lasso = {
                    'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100] 
                   }

grid_search_lasso = GridSearchCV(pipeline_multi_lasso, param_grid_lasso, cv=10, scoring='neg_mean_squared_error', n_jobs = -1)
grid_search_lasso.fit(X_train, y_train)

In [ ]:
print("Best hyperparameters:", grid_search_lasso.best_params_)
print("Best score:", grid_search_lasso.best_score_)

In [ ]:
best_lasso_model = grid_search_lasso.best_estimator_

In [ ]:
save_model(model = best_lasso_model, model_name = 'Lasso')

In [ ]:
lasso_predictions = best_lasso_model.predict(X_test)

In [ ]:
plot_cv_scores(model_pipeline = best_lasso_model, model_type = 'Multi-task Lasso', cv_fold = 10)

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = lasso_predictions, model_type = 'Multi-task Lasso')

In [ ]:
get_r2_cross_val(model = best_lasso_model, model_name = 'Lasso')

In [ ]:
get_metrics(y_predicted = lasso_predictions)

In [ ]:
get_spearman_correlation(model_predictions = lasso_predictions, model_name = 'Lasso')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = lasso_predictions, model_name = 'Lasso')

In [ ]:
plot_r2_per_metabolite(best_lasso_model, "Lasso", X_test, y_test)

In [ ]:
r2_lasso = get_r2_per_metabolite(best_lasso_model, 'Lasso', X_test, y_test)
r2_lasso

In [ ]:
plot_spearman_per_metabolite(best_lasso_model, 'Lasso', X_test, y_test)

In [ ]:
spearman_lasso = get_spearman_per_metabolite(best_lasso_model, 'Lasso', X_test, y_test)
spearman_lasso

In [ ]:
r2_lasso.to_csv(f'{results_path}/r2_lasso.csv', index = False)
spearman_lasso.to_csv(f'{results_path}/spearman_lasso.csv', index = False)

# Multi-task Elastic-Net

In [ ]:
pipeline_elasticnet = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('pca', PCA(0.95)),
    ('model', MultiTaskElasticNet(random_state = 0))
])

In [ ]:
param_grid_elasticnet = {
    'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0.1, 0.5, 0.9]
}


In [ ]:
grid_search_elasticnet = GridSearchCV(
                                      pipeline_elasticnet,
                                      param_grid_elasticnet,
                                      cv=10,
                                      scoring='neg_mean_squared_error',
                                      n_jobs = -1
                                      )
grid_search_elasticnet.fit(X_train, y_train)

In [ ]:
print("Best hyperparameters for ElasticNet:", grid_search_elasticnet.best_params_)
print("Best score for ElasticNet:", grid_search_elasticnet.best_score_)

In [ ]:
best_elasticnet_model = grid_search_elasticnet.best_estimator_

In [ ]:
save_model(model = best_elasticnet_model, model_name = 'ElasticNet')

In [ ]:
elasticnet_predictions = best_elasticnet_model.predict(X_test)

In [ ]:
plot_cv_scores(model_pipeline = best_elasticnet_model, model_type = 'Multi-task Elastic Net (All metabolites)', cv_fold = 10)

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = elasticnet_predictions, model_type = 'Multi-task Elastic Net (All metabolites)')

In [ ]:
get_r2_cross_val(model = best_elasticnet_model, model_name = 'ElasticNet')

In [ ]:
get_metrics(y_predicted = elasticnet_predictions)

In [ ]:
get_spearman_correlation(model_predictions = elasticnet_predictions, model_name = 'ElasticNet')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = elasticnet_predictions, model_name = 'ElasticNet')

In [ ]:
plot_r2_per_metabolite(best_elasticnet_model, "ElasticNet", X_test, y_test)

In [ ]:
y_test.to_csv(f'{results_path}/y_test_elasticnet_model.csv', index = True)

In [ ]:
elasticnet_predictions = pd.DataFrame(elasticnet_predictions, index = y_test.index, columns = y_test.columns)
elasticnet_predictions

In [ ]:
elasticnet_predictions.to_csv(f'{results_path}/predictions_elasticnet_model.csv', index = True)

In [ ]:
r2_elastic = get_r2_per_metabolite(best_elasticnet_model, 'ElasticNet', X_test, y_test)
r2_elastic

In [ ]:
plot_spearman_per_metabolite(best_elasticnet_model, 'ElasticNet', X_test, y_test)

In [ ]:
spearman_elastic = get_spearman_per_metabolite(best_elasticnet_model, 'ElasticNet', X_test, y_test)
spearman_elastic

In [ ]:
r2_elastic.to_csv(f'{results_path}/r2_elastic.csv', index = False)
spearman_elastic.to_csv(f'{results_path}/spearman_elastic.csv', index = False)

# Random Forest Regressor

In [ ]:
pipeline_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('pca', PCA(0.95)),
    ('model', MultiOutputRegressor(RandomForestRegressor(n_estimators = 500, max_depth = 20, random_state=0, n_jobs = -1)))
])

In [ ]:
model_rf = pipeline_rf.fit(X_train, y_train)

In [ ]:
# pipeline_rf = Pipeline([
#     ('imputer', SimpleImputer(strategy='mean')),
#     ('scaler', StandardScaler()),
#     ('pca', PCA(0.95)),
#     ('model', MultiOutputRegressor(RandomForestRegressor(random_state=0, n_jobs = -1), n_jobs = -1))
# ])

In [ ]:
# rf_param_grid = {
#     'model__estimator__n_estimators' : [100, 150, 200, 250, 300, 500],
#     'model__estimator__max_depth': [5, 10, 20, 30, 50]
# }

In [ ]:
# grid_search_rf = GridSearchCV(
#                               pipeline_rf,
#                               rf_param_grid,
#                               cv = 5,
#                               scoring = 'neg_mean_squared_error',
#                               verbose = 3,
#                               n_jobs = -1
#                               )

In [ ]:
# grid_search_rf.fit(X_train, y_train)

In [ ]:
# print("Best hyperparameters:", grid_search_rf.best_params_)
# print("Best score:", grid_search_rf.best_score_)

In [ ]:
# model_rf = grid_search_rf.best_estimator_

In [ ]:
save_model(model = model_rf, model_name = 'Random Forest Regressor')

In [ ]:
predictions_rf = model_rf.predict(X_test)

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = predictions_rf, model_type = 'Random Forest Regressor')

In [ ]:
plot_cv_scores(model_pipeline = model_rf, model_type = 'Multi-output Random Forest Regressor', cv_fold = 10)

In [ ]:
get_metrics(y_predicted = predictions_rf)

In [ ]:
get_r2_cross_val(model = model_rf, model_name = 'Random Forest Regressor')

In [ ]:
get_spearman_correlation(model_predictions = predictions_rf, model_name = 'Random Forest Regressor')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = predictions_rf, model_name = 'Random Forest Regressor')

In [ ]:
plot_r2_per_metabolite(model_rf, "Random Forest Regressor", X_test, y_test)

In [ ]:
r2_rf = get_r2_per_metabolite(model_rf, "Random Forest Regressor", X_test, y_test)
r2_rf

In [ ]:
plot_spearman_per_metabolite(model_rf, "Random Forest Regressor", X_test, y_test)

In [ ]:
spearman_rf = get_spearman_per_metabolite(model_rf, "Random Forest Regressor", X_test, y_test)
spearman_rf

In [ ]:
r2_rf.to_csv(f'{results_path}/r2_rf.csv', index = False)
spearman_rf.to_csv(f'{results_path}/spearman_rf.csv', index = False)

# XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor

pipeline_xgb = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('pca', PCA(0.95)),
    ('model', MultiOutputRegressor(XGBRegressor(n_estimators = 100, max_depth = 5, subsample=1.0, random_state=0, n_jobs = -1), n_jobs = -1))
])

model_xgb = pipeline_xgb.fit(X_train, y_train)

In [ ]:
# pipeline_xgb = Pipeline([
#     ('imputer', SimpleImputer(strategy='mean')),
#     ('scaler', StandardScaler()),
#     ('pca', PCA(0.95)),
#     ('model', MultiOutputRegressor(XGBRegressor(random_state=0, tree_method = "hist", device = "cuda", predictor='auto', n_jobs = -1)))
# ])

In [ ]:
# xgb_param_grid = {
#     'model__estimator__n_estimators': [100, 200, 300],
#     'model__estimator__max_depth': [5, 10, 20],
#     'model__estimator__subsample': [0.5, 1.0]
# }



# xgb_grid_search = GridSearchCV(
#                                estimator=pipeline_xgb,
#                                param_grid=xgb_param_grid,
#                                scoring='neg_mean_squared_error',
#                               #  cv=5,
#                                verbose=3,
#                                n_jobs=-1
#                                )

In [ ]:
# xgb_grid_search.fit(X_train, y_train)

In [ ]:
# print("Best Parameters:", xgb_grid_search.best_params_)
# print("Best Score:", xgb_grid_search.best_score_)

In [ ]:
# model_xgb = xgb_grid_search.best_estimator_

In [ ]:
save_model(model = model_xgb, model_name = 'XGBoost Regressor')

In [ ]:
predictions_xgb = model_xgb.predict(X_test)

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = predictions_xgb, model_type = 'XGBoost Regressor')

In [ ]:
plot_cv_scores(model_pipeline = model_xgb, model_type = 'Multi-output XGBoost Regressor', cv_fold = 10)

In [ ]:
get_r2_cross_val(model = model_xgb, model_name = 'XGBoost Regressor')

In [ ]:
get_metrics(y_predicted = predictions_xgb)

In [ ]:
get_spearman_correlation(model_predictions = predictions_xgb, model_name = 'XGBoost Regressor')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = predictions_xgb, model_name = 'XGBoost Regressor')

In [ ]:
plot_r2_per_metabolite(model_xgb, "XGBoost Regressor", X_test, y_test)

In [ ]:
r2_xgb = get_r2_per_metabolite(model_xgb, "XGBoost Regressor", X_test, y_test)
r2_xgb

In [ ]:
plot_spearman_per_metabolite(model_xgb, "XGBoost Regressor", X_test, y_test)

In [ ]:
spearman_xgb = get_spearman_per_metabolite(model_xgb, "XGBoost Regressor", X_test, y_test)
spearman_xgb

In [ ]:
r2_xgb.to_csv(f'{results_path}/r2_xgb.csv', index = False)
spearman_xgb.to_csv(f'{results_path}/spearman_xgb.csv', index = False)

# LightGBM Regressor

In [ ]:
pipeline_lgbm = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('pca', PCA(0.95)),
    ('model', MultiOutputRegressor(LGBMRegressor(n_estimators = 100, max_depth= 5, num_leaves = 30, random_state=0, n_jobs = -1)))
])

best_lgbm_model = pipeline_lgbm.fit(X_train, y_train)

In [ ]:
# pipeline_lgbm = Pipeline([
#     ('imputer', SimpleImputer(strategy='mean')),
#     ('scaler', StandardScaler()),
#     ('pca', PCA(0.95)),
#     ('model', MultiOutputRegressor(LGBMRegressor(random_state=0, n_jobs = -1), n_jobs = -1))
# ])

# lgbm_param_grid = {
#     'model__estimator__n_estimators': [100, 150, 200, 250, 300, 500],
#     'model__estimator__max_depth': [5, 10, 20, 30, 50],
#     'model__estimator__num_leaves': [30, 50, 100, 200]
# }

# grid_search_lgbm = GridSearchCV(
#     estimator=pipeline_lgbm,
#     param_grid=lgbm_param_grid,
#     scoring='neg_mean_squared_error',
#     # cv=5,
#     verbose=3,
#     n_jobs=-1
# )

In [ ]:
# grid_search_lgbm.fit(X_train, y_train)

In [ ]:
# print("Best Parameters:", grid_search_lgbm.best_params_)
# print("Best Score:", grid_search_lgbm.best_score_)

In [ ]:
# best_lgbm_model = grid_search_lgbm.best_estimator_

In [ ]:
save_model(model = best_lgbm_model, model_name = 'LightGBM Regressor')

In [ ]:
predictions_lgbm = best_lgbm_model.predict(X_test)

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = predictions_lgbm, model_type = 'LightGBM Regressor')

In [ ]:
plot_cv_scores(model_pipeline = best_lgbm_model, model_type = 'Multi-output LightGBM Regressor', cv_fold = 10)

In [ ]:
get_r2_cross_val(model = best_lgbm_model, model_name = 'LightGBM Regressor')

In [ ]:
get_metrics(y_predicted = predictions_lgbm)

In [ ]:
get_spearman_correlation(model_predictions = predictions_lgbm, model_name = 'LightGBM Regressor')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = predictions_lgbm, model_name = 'LightGBM Regressor')

In [ ]:
plot_r2_per_metabolite(best_lgbm_model, "LightGBM Regressor", X_test, y_test)

In [ ]:
r2_lgbm = get_r2_per_metabolite(best_lgbm_model, "LightGBM Regressor", X_test, y_test)
r2_lgbm

In [ ]:
plot_spearman_per_metabolite(best_lgbm_model, "LightGBM Regressor", X_test, y_test)

In [ ]:
spearman_lgbm = get_spearman_per_metabolite(best_lgbm_model, "LightGBM Regressor", X_test, y_test)
spearman_lgbm

In [ ]:
r2_lgbm.to_csv(f'{results_path}/r2_lgbm.csv', index = False)
spearman_lgbm.to_csv(f'{results_path}/spearman_lgbm.csv', index = False)

# MLP Regressor

In [ ]:
from sklearn.neural_network import MLPRegressor

In [ ]:
pipeline_mlp = Pipeline([
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('scaler', StandardScaler()),
    ('pca', PCA(0.95)),
    ('model', MLPRegressor(activation = 'tanh',
                           alpha = 10,
                           batch_size = 64,
                           early_stopping = True,
                           hidden_layer_sizes = (50, 50),
                           solver = 'sgd',
                           random_state = 0,
                           learning_rate = 'adaptive'))
])

best_mlp_model = pipeline_mlp.fit(X_train, y_train)

In [ ]:
# pipeline_mlp = Pipeline([
#     ('imputer', SimpleImputer(strategy = 'mean')),
#     ('scaler', StandardScaler()),
#     ('pca', PCA(0.95)),
#     ('model', MLPRegressor(random_state = 0, learning_rate = 'adaptive'))
# ])


# param_grid_mlp = {
#     'model__hidden_layer_sizes': [(100,), (50, 50), (100, 50, 25)],
#     'model__activation': ['relu', 'tanh'],
#     'model__solver': ['adam', 'sgd'],
#     'model__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
#     'model__batch_size': [32, 64, 128],
#     'model__early_stopping': [True, False]
# }


# grid_search_mlp = GridSearchCV(
#     estimator = pipeline_mlp,
#     param_grid = param_grid_mlp,
#     scoring = 'neg_mean_squared_error',
#     verbose = 3,
#     n_jobs = -1
# )

# grid_search_mlp.fit(X_train, y_train)

In [ ]:
# print("Best Parameters:", grid_search_mlp.best_params_)
# print("Best Score:", grid_search_mlp.best_score_)

In [ ]:
# best_mlp_model = grid_search_mlp.best_estimator_

In [ ]:
save_model(model = best_mlp_model, model_name = 'MLP Regressor')

In [ ]:
predictions_mlp = best_mlp_model.predict(X_test)

In [ ]:
get_r2_mse_rmse_plot_pred(data_y_test = y_test, model_predictions = predictions_mlp, model_type = 'MLP Regressor')

In [ ]:
plot_cv_scores(model_pipeline = best_mlp_model, model_type = 'MLP Regressor', cv_fold = 10)

In [ ]:
get_r2_cross_val(model = best_mlp_model, model_name = 'MLP Regressor')

In [ ]:
get_metrics(y_predicted = predictions_mlp)

In [ ]:
get_spearman_correlation(model_predictions = predictions_mlp, model_name = 'MLP Regressor')

In [ ]:
get_residuals_vs_pred_plot(model_predictions = predictions_mlp, model_name = 'MLP Regressor')

In [ ]:
plot_r2_per_metabolite(best_mlp_model, "MLP Regressor", X_test, y_test)

In [ ]:
r2_mlp = get_r2_per_metabolite(best_mlp_model, "MLP Regressor", X_test, y_test)
r2_mlp

In [ ]:
plot_spearman_per_metabolite(best_mlp_model, "MLP Regressor", X_test, y_test)

In [ ]:
spearman_mlp = get_spearman_per_metabolite(best_mlp_model, "MLP Regressor", X_test, y_test)
spearman_mlp

In [ ]:
r2_mlp.to_csv(f'{results_path}/r2_mlp.csv', index = False)
spearman_mlp.to_csv(f'{results_path}/spearman_mlp.csv', index = False)